# MINOS U-View Sparse CNN (MIT TorchSparse Engine)

This notebook trains a **simple single-view (U-view)** sparse convolutional neural network (`SimpleUViewSparseCNN`) to classify MINOS **Charged Current (CC)** vs **Neutral Current (NC)** event displays using MIT TorchSparse API (`SparseTensor` and `torchsparse.nn`).

All hyperparameters, model channels, learning rates, and scheduler settings are fully configurable in the **Global Configuration** block below.

In [ ]:
# Setup, Imports, and Module Reloading
import sys
import importlib
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

# Purge any stale in-memory cached imports for src
for mod in list(sys.modules.keys()):
    if mod == 'src' or mod.startswith('src.'):
        del sys.modules[mod]

import src
importlib.reload(src)

from src import (
    SparseTensor,
    MINOSSingleViewDataset,
    create_uview_dataloaders,
    SimpleUViewSparseCNN,
    train_model,
    validate_epoch
)
import src.torchsparse.nn as spnn

print('Imports and TorchSparse API loaded successfully!')

## 1. Global Hyperparameters & Experiment Configuration

In [ ]:
# ==============================================================================
# Global Configuration & Hyperparameters
# ==============================================================================
CONFIG = {
    # Dataset Parameters
    "root_filepath": "/home/philip/UCL/minos/f21048000_0000_L010185N_D07_r3.sntp.dogwood5.0.root",
    "target_view": 2,          # 2 = U-View
    "max_events": 12000,       # Set to None to load all events in file
    "batch_size": 32,
    "val_split": 0.20,
    "random_seed": 42,

    # Model Architecture
    "conv_channels": [16, 32],
    "fc_dims": [16],
    "dropout": 0.1,

    # Optimization & Learning Rate Scheduler
    "num_epochs": 10,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "step_size": 3,            # Decays LR every step_size epochs
    "gamma": 0.1,              # Multiplicative LR decay factor
    "use_class_weights": True  # Enable inverse class weighting in loss
}

# Set random seeds for reproducibility
torch.manual_seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using PyTorch device: {device}')
print('Experiment Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## 2. Load U-View MINOS Dataset & DataLoaders

In [ ]:
dataset = MINOSSingleViewDataset(
    root_filepath=CONFIG["root_filepath"],
    max_events=CONFIG["max_events"],
    target_view=CONFIG["target_view"]
)

train_loader, val_loader, _, _ = create_uview_dataloaders(
    dataset,
    batch_size=CONFIG["batch_size"],
    val_split=CONFIG["val_split"],
    random_seed=CONFIG["random_seed"]
)

if CONFIG["use_class_weights"]:
    class_weights = dataset.get_class_weights(device=device)
    print(f'Class Weights -> NC (0): {class_weights[0]:.2f} | CC (1): {class_weights[1]:.2f}')
else:
    class_weights = None
    print('Class Weights -> Disabled (Unweighted CrossEntropy)')

print(f'Total Dataset size: {len(dataset)} U-view events')

## 3. Model Initialization & Training (`SimpleUViewSparseCNN`)

In [ ]:
model = SimpleUViewSparseCNN(
    in_channels=1,
    conv_channels=CONFIG["conv_channels"],
    fc_dims=CONFIG["fc_dims"],
    dropout=CONFIG["dropout"]
)

print(f'SimpleUViewSparseCNN Total Trainable Parameters: {model.get_num_params():,}')

history, best_metrics = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=CONFIG["num_epochs"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    step_size=CONFIG["step_size"],
    gamma=CONFIG["gamma"],
    class_weights=class_weights,
    device=device,
    verbose=True
)

## 4. Training & Validation Performance Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='navy')
ax1.plot(epochs, history['val_loss'], 's--', label='Val Loss', color='crimson')
ax1.set_title('Cross Entropy Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

# Accuracy & F1 curves
ax2.plot(epochs, [a * 100 for a in history['val_acc']], 'o-', label='Val Accuracy (%)', color='forestgreen')
ax2.plot(epochs, [f * 100 for f in history['val_f1']], 's--', label='Val F1 Score (%)', color='darkorange')
ax2.plot(epochs, [a * 100 for a in history['val_auc']], '^:', label='Val ROC-AUC (%)', color='purple')
ax2.set_title('Validation Classification Metrics')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Percentage (%)')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()